# Unificar la base de rasgos funcionales de Oyacachi + Guacamayos

Parte de 2 archivos, uno por sitio:
- **`01__Rasgos_Oyacachi.xlsx`** (hoja `Hoja 1`): 33 árboles. Mide el grosor de hoja con 30 puntos por árbol (columnas `1`...`30`, sin separar por hoja).
- **`02__Rasgos_Guacamayos.xlsx`** (hoja `Hoja 1`): 28 árboles. Mide el grosor de hoja con hasta 20 puntos por cada una de 3 hojas (columnas `T1_1`...`T3_20`).

**Diferencia clave con Sumaco/Galeras/Selva Viva: todavía no se tomó muestra de madera aquí**, así que no hay `crust thickness`, `wet lenght cm`, `wet wood weight` ni `dry wood weight` en ninguno de los 2 archivos. Eso significa que **wood density, WSG, stem water content y force to punch no se pueden calcular todavía** — quedan como columnas `NaN`, listas para completarse cuando se haga el muestreo de madera. Tampoco hay peso fresco de hoja en casi ningún árbol (solo 3 de 61 en total), así que `LDMC` casi no se puede calcular.

**Qué se puede calcular con lo que sí hay:** `SLA` (área foliar / peso seco) y el grosor medio de hoja (promediando todos los puntos medidos), que sirven además como control cruzado de los valores que ya venían precalculados en los archivos originales.

## 1. Librerías

In [ ]:
import pandas as pd
import numpy as np
import re
import openpyxl
from openpyxl.styles import PatternFill

pd.set_option('display.max_columns', None)

## 2. Extraer las celdas resaltadas en amarillo

Igual que en los notebooks anteriores (Galeras, Selva Viva): se recorre cada archivo con `openpyxl` y se guarda qué celdas tienen relleno amarillo puro (`FFFFFF00`).

In [ ]:
def extraer_resaltado_amarillo(path, sheet_name):
    wb = openpyxl.load_workbook(path)
    ws = wb[sheet_name]
    headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
    resaltado = {}
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            fill = cell.fill
            if fill and fill.fgColor and fill.fgColor.type == 'rgb' and fill.fgColor.rgb == 'FFFFFF00':
                col_name = headers[cell.column - 1]
                resaltado.setdefault(cell.row - 2, set()).add(col_name)
    return resaltado

resaltado_oya = extraer_resaltado_amarillo('01__Rasgos_Oyacachi.xlsx', 'Hoja 1')
resaltado_gua = extraer_resaltado_amarillo('02__Rasgos_Guacamayos.xlsx', 'Hoja 1')
print('Filas con amarillo Oyacachi:', len(resaltado_oya))
print('Filas con amarillo Guacamayos:', len(resaltado_gua))

## 3. Nota sobre la metodología de campo (hoja `Hoja 2` de Oyacachi)

El archivo de Oyacachi trae una segunda hoja con una nota metodológica: *"Para los que tienen problemas se calcula el promedio del peso de 1 hoja y se resta del total"* — es decir, cuando el peso de un lote de hojas parecía inconsistente, corregían restando el peso promedio de 1 hoja. Esto es relevante para 2 filas de `Guacamayos` marcadas con el comentario **"1 hoja pesada demás"**, donde a pesar de esa corrección el peso resultante sigue sin ser plausible (ver sección de control de calidad más abajo).

## 4. Función auxiliar

In [ ]:
def a_numero(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return np.nan

## 5. Cargar y estandarizar Oyacachi

Las 30 columnas de grosor (nombradas `1` a `30`, sin distinguir hoja) se promedian directamente en `mean_leaf_thickness_mm` y se descartan como columnas individuales (quedan resumidas en el promedio, ya no aportan por separado).

In [ ]:
oya = pd.read_excel('01__Rasgos_Oyacachi.xlsx', sheet_name='Hoja 1')
thickness_cols_oya = [c for c in oya.columns if isinstance(c, int)]

ren_oya = {
    'Plot': 'Plot', 'Site': 'Site', 'Plot-code': 'PlotID', 'Sub': 'Subplot', 'altitude': 'altitude_m',
    'Date ': 'sampling_date', 'TreeID_2025': 'new_tree_ID_2025', 'oldTreeID': 'treeID',
    'Family': 'family', 'genus': 'genus', 'specie': 'species',
    '# leaves': 'n_leaves', 'Leaf fresh weight (g)': 'leaf_fresh_weight_g', 'Leaf dry weight (g)': 'leaf_dry_weight_g',
    '# leaves for area': 'n_leaves_area', 'Leaf area (cm²)': 'leaf_area_cm2', 'SLA (cm2/g)': 'SLA_original',
    'mean_thickness_(mm)': 'mean_thickness_original', 'herb_code': 'herb_code',
}
oya = oya.rename(columns=ren_oya)
oya['mean_leaf_thickness_mm'] = oya[thickness_cols_oya].mean(axis=1)
oya = oya.drop(columns=thickness_cols_oya)
oya['source_file'] = '01__Rasgos_Oyacachi.xlsx'
oya['_orig_row'] = range(len(oya))
oya.shape

## 6. Cargar y estandarizar Guacamayos

Aquí el grosor sí distingue hoja (`T1_`, `T2_`, `T3_` + hasta 20 puntos cada una) — igual se promedian todas juntas en `mean_leaf_thickness_mm`.

In [ ]:
gua = pd.read_excel('02__Rasgos_Guacamayos.xlsx', sheet_name='Hoja 1')
thickness_cols_gua = [c for c in gua.columns if re.match(r'^T[123]_\d+$', str(c))]

ren_gua = {
    'Plot': 'Plot', 'Site': 'Site', 'Plot-code': 'PlotID', 'Sub': 'Subplot', 'altitud': 'altitude_m',
    'Date ': 'sampling_date', 'TreeID_2025': 'new_tree_ID_2025', 'oldTreeID': 'treeID',
    'Family': 'family', 'genus': 'genus', 'specie': 'species',
    '# leaves scaned': 'n_leaves_area', '# leaves weighted': 'n_leaves', 'Leaf fresh weight (g)': 'leaf_fresh_weight_g',
    'Leaf dry weight (g)': 'leaf_dry_weight_g', 'Leaf area (cm²)': 'leaf_area_cm2', 'SLA (cm2/g)': 'SLA_original',
    'comment': 'comment', 'mean_thickness_(mm)': 'mean_thickness_original',
}
gua = gua.rename(columns=ren_gua)
gua['mean_leaf_thickness_mm'] = gua[thickness_cols_gua].mean(axis=1)
gua = gua.drop(columns=thickness_cols_gua)
gua['source_file'] = '02__Rasgos_Guacamayos.xlsx'
gua['_orig_row'] = range(len(gua))
gua.shape

## 7. Unificar

Son 2 sitios distintos sin árboles en común, así que simplemente se concatenan.

In [ ]:
unificado = pd.concat([oya, gua], ignore_index=True, sort=False)
unificado['QC_flag'] = ''
for c in ['leaf_fresh_weight_g', 'leaf_dry_weight_g', 'leaf_area_cm2']:
    unificado[c] = unificado[c].apply(a_numero)

print('Base unificada:', unificado.shape)
unificado['Site'].value_counts()

## 8. Calcular los rasgos que sí son posibles

- **SLA (cm²/g)** = área foliar / peso seco de hoja — se recalcula desde cero como control cruzado del valor original.
- **LDMC (mg/g)** = peso seco × 1000 / peso fresco — solo calculable en los 3 árboles que sí tienen peso fresco registrado.
- **Wood density, WSG, stem water content, force to punch**: no calculables (sin muestreo de madera todavía) — quedan en `NaN`.

In [ ]:
unificado['SLA_cm2_g'] = unificado['leaf_area_cm2'] / unificado['leaf_dry_weight_g']
unificado['LDMC_mg_g'] = unificado['leaf_dry_weight_g'] * 1000 / unificado['leaf_fresh_weight_g']

for c in ['wood_density_g_cm3', 'WSG', 'stem_water_content_pct', 'force_to_punch_kN_m']:
    unificado[c] = np.nan

print('Arboles con LDMC calculable (tienen peso fresco):', unificado['LDMC_mg_g'].notna().sum(), 'de', len(unificado))
unificado[['SLA_cm2_g', 'SLA_original', 'mean_leaf_thickness_mm', 'LDMC_mg_g']].describe().T

## 9. Revisar rangos y notas de campo

Se marca cualquier fila con el comentario **"1 hoja pesada demás"** (la corrección mencionada en la `Hoja 2` de Oyacachi parece no haber quedado bien aplicada en 2 casos: uno con un peso seco de 513 g para una hoja de solo 115 cm² de área, y otro con un peso fresco de 1712 g — ambos fisicamente imposibles).

In [ ]:
rangos = {
    'SLA_cm2_g': (10, 500),
    'mean_leaf_thickness_mm': (0.05, 1.0),
    'LDMC_mg_g': (50, 650),
}
for col, (lo, hi) in rangos.items():
    fuera = unificado[col].notna() & ((unificado[col] < lo) | (unificado[col] > hi))
    unificado.loc[fuera, 'QC_flag'] += f'{col} fuera de rango [{lo}-{hi}]; '

tiene_nota_pesada = unificado['comment'].notna() & unificado['comment'].astype(str).str.contains('pesada', case=False)
unificado.loc[tiene_nota_pesada, 'QC_flag'] += (
    'Nota de campo: "1 hoja pesada demas" -- revisar si la correccion de peso (ver hoja "Hoja 2" '
    'del archivo original de Oyacachi) se aplico bien, el peso resultante no es plausible; ')

sin_taxo = unificado['family'].isna() & unificado['genus'].isna() & unificado['species'].isna()
unificado.loc[sin_taxo, 'QC_flag'] += 'sin familia/genero/especie; '

sin_id = unificado['treeID'].isna() & unificado['new_tree_ID_2025'].isna()
unificado.loc[sin_id, 'QC_flag'] += 'sin treeID ni new_tree_ID_2025; '

print('Filas marcadas:', (unificado['QC_flag'] != '').sum(), 'de', len(unificado))
unificado[unificado['QC_flag'] != ''][['Site', 'PlotID', 'new_tree_ID_2025', 'genus', 'species', 'QC_flag']]

## 10. Reordenar columnas

In [ ]:
front = ['Site', 'PlotID', 'Plot', 'Subplot', 'treeID', 'new_tree_ID_2025',
         'family', 'genus', 'species', 'sampling_date', 'altitude_m', 'herb_code',
         'n_leaves', 'n_leaves_area', 'leaf_fresh_weight_g', 'leaf_dry_weight_g', 'leaf_area_cm2',
         'SLA_cm2_g', 'SLA_original', 'LDMC_mg_g', 'mean_leaf_thickness_mm', 'mean_thickness_original',
         'wood_density_g_cm3', 'WSG', 'stem_water_content_pct', 'force_to_punch_kN_m',
         'comment', 'QC_flag', 'source_file', '_orig_row']
otras = [c for c in unificado.columns if c not in front]
unificado = unificado[front + otras]
unificado.shape

## 11. Guardar y volver a aplicar el resaltado amarillo

Igual que en los notebooks anteriores. Como las decenas de columnas de grosor individuales (`1`...`30`, `T1_1`...`T3_20`) se colapsaron en una sola columna `mean_leaf_thickness_mm`, cualquier resaltado que caía en esas columnas se traslada también a `mean_leaf_thickness_mm` en la tabla final.

In [ ]:
out_path = 'Rasgos_Oyacachi_Guacamayos_unificado.xlsx'
reporte = unificado[unificado['QC_flag'] != ''][
    ['Site', 'PlotID', 'treeID', 'new_tree_ID_2025', 'family', 'genus', 'species', 'QC_flag', 'source_file']
]

with pd.ExcelWriter(out_path, engine='openpyxl') as writer:
    unificado.to_excel(writer, sheet_name='Rasgos_OYA_GUA_unificado', index=False)
    reporte.to_excel(writer, sheet_name='Filas_a_revisar', index=False)

wb = openpyxl.load_workbook(out_path)
ws = wb['Rasgos_OYA_GUA_unificado']
headers = [c.value for c in next(ws.iter_rows(min_row=1, max_row=1))]
col_idx = {h: i + 1 for i, h in enumerate(headers)}
amarillo = PatternFill(start_color='FFFFFF00', end_color='FFFFFF00', fill_type='solid')

for i, row in unificado.reset_index(drop=True).iterrows():
    origen = row['source_file']
    orig_row = row['_orig_row']
    resaltado_orig = resaltado_oya if origen.startswith('01') else resaltado_gua
    ren_dict = ren_oya if origen.startswith('01') else ren_gua
    cols_resaltadas = resaltado_orig.get(orig_row, set())
    for orig_col in cols_resaltadas:
        nuevo_col = ren_dict.get(orig_col, orig_col)
        if nuevo_col in col_idx:
            ws.cell(row=i + 2, column=col_idx[nuevo_col]).fill = amarillo
        elif orig_col in thickness_cols_oya or str(orig_col) in [str(x) for x in thickness_cols_gua]:
            if 'mean_leaf_thickness_mm' in col_idx:
                ws.cell(row=i + 2, column=col_idx['mean_leaf_thickness_mm']).fill = amarillo

wb.save(out_path)
print('Guardado con resaltado:', out_path)

## 12. Resumen y sugerencias de revisión manual

- **Sin muestreo de madera todavía**: `wood density`, `WSG`, `stem water content` y `force to punch` quedan en `NaN` para las 61 filas — se podrán calcular con las mismas fórmulas ya usadas en Sumaco/Galeras/Selva Viva en cuanto exista el muestreo de madera de Oyacachi y Guacamayos.
- **Casi no hay peso fresco de hoja** (solo 3 de 61 árboles) — por eso LDMC queda casi todo en `NaN`; si se retoma esa medición en campo, se puede completar después.
- **2 árboles con el comentario "1 hoja pesada demás"** cuyos valores siguen siendo físicamente imposibles después de la corrección mencionada en la `Hoja 2` de Oyacachi — revisar el peso original antes de la corrección para recalcularlos bien: `GUA07` (TreeID 8147, *Chomelia tenuiflora*, peso seco 513 g para una hoja de 115 cm²) y `GUA55` (TreeID 8122, *Ruagea glabra*, peso fresco de 1712 g).
- **2 árboles de Oyacachi con LDMC entre 900-909 mg/g** (`OYC-83` ID 4240 y `OYC-81` ID 8056, ambos *Gynoxis acostae*) — valores casi sin agua, poco plausibles para una hoja fresca; revisar si el peso fresco se tomó correctamente.
- El resaltado amarillo original se conservó (31 celdas); confirma con el equipo de campo qué significaba antes de asumir que siempre es "dato dudoso".